In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as ticker
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'font.family': 'sans-serif',
    'axes.spines.top': False,
    'axes.spines.right': False,
})

FIGURES = Path('../outputs/figures')
FIGURES.mkdir(parents=True, exist_ok=True)
THRESHOLD = 50

df = pd.read_csv('../Data:/grade_level_gaps.csv')
df['grade'] = df['grade'].astype(int)

# Restrict to named-province rows only (exclude Gender: * slices)
PROVINCE_ROWS = [p for p in df['province'].unique() if not p.startswith('Gender')]
prov_df = df[df['province'].isin(PROVINCE_ROWS)].copy()

print(f'Province rows: {PROVINCE_ROWS}')
print(f'Grades: {sorted(prov_df["grade"].unique())}')

## Figure A — Sample Size Heatmap: Private Students per Province × Grade

Before relying on any province-level gap estimate, it is essential to verify that the underlying private-school sample is large enough for the statistic to be reliable. This heatmap shows `n_private` — the number of private school students assessed — for every province × grade cell. Cells with **fewer than 50 students** are flagged in red and cross-hatched; estimates from those cells should be treated with caution or excluded from analysis.

In [ ]:
piv = prov_df.pivot(index='province', columns='grade', values='n_private')

# Sort provinces: National first, then alphabetically
ordered = ['National'] + sorted([p for p in piv.index if p != 'National'])
piv = piv.loc[ordered]

flag_mask = (piv < THRESHOLD) & piv.notna()

# Build annotation strings
annot_df = piv.copy().astype(object)
for row in piv.index:
    for col in piv.columns:
        v = piv.loc[row, col]
        annot_df.loc[row, col] = f'{int(v):,}' if pd.notna(v) else ''

fig, ax = plt.subplots(figsize=(13, 5))

sns.heatmap(
    piv,
    annot=annot_df,
    fmt='',
    cmap='Blues',
    vmin=0,
    linewidths=0.6,
    linecolor='white',
    cbar_kws={'label': 'Private students (n)', 'shrink': 0.75},
    ax=ax,
)

# Overlay red hatch on flagged cells
for r_idx, row in enumerate(piv.index):
    for c_idx, col in enumerate(piv.columns):
        if flag_mask.loc[row, col]:
            ax.add_patch(
                plt.Rectangle(
                    (c_idx, r_idx), 1, 1,
                    fill=True, facecolor='#ffcccc', alpha=0.55,
                    hatch='////', edgecolor='#cc0000',
                    linewidth=0, zorder=3,
                )
            )

# Re-draw annotation text on top of the overlay for flagged cells
for text_obj in ax.texts:
    x_center, y_center = text_obj.get_position()
    # heatmap text positions are in data coords: (col+0.5, row+0.5)
    c_idx = int(x_center)
    r_idx = int(y_center)
    if 0 <= r_idx < len(piv.index) and 0 <= c_idx < len(piv.columns):
        row = piv.index[r_idx]
        col = piv.columns[c_idx]
        if flag_mask.loc[row, col]:
            text_obj.set_color('#990000')
            text_obj.set_fontweight('bold')
            text_obj.set_zorder(4)

ax.set_title(f'Private School Sample Size per Province × Grade  (red = n < {THRESHOLD})',
             fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Grade', labelpad=8)
ax.set_ylabel('')
ax.tick_params(axis='y', rotation=0)

legend_patch = mpatches.Patch(
    facecolor='#ffcccc', edgecolor='#cc0000', hatch='////', alpha=0.8,
    label=f'n < {THRESHOLD} (unreliable estimate)',
)
ax.legend(handles=[legend_patch], loc='lower right',
          bbox_to_anchor=(1.0, -0.22), frameon=False, fontsize=10)

plt.tight_layout()
fig.savefig(FIGURES / '02_sample_size_heatmap.png', bbox_inches='tight')
fig.savefig(FIGURES / '02_sample_size_heatmap.pdf', bbox_inches='tight')
plt.show()

print(f'\nFlagged cells (n_private < {THRESHOLD}):')
flagged = flag_mask.stack()
print(flagged[flagged].reset_index().rename(columns={'grade': 'Grade', 'province': 'Province', 0: 'Flagged'})
      [['Province', 'Grade']].assign(n_private=lambda d: d.apply(
          lambda r: int(piv.loc[r['Province'], r['Grade']]), axis=1)).to_string(index=False))

## Figure 1b — Govt–Private Arithmetic Gap by Grade (n_private ≥ 50 only)

This chart reproduces Figure 1 from `01_gap_by_grade.ipynb` with one modification: any province × grade data point where the private school sample falls below 50 students is dropped before plotting, causing the corresponding line segment to break. This prevents visually prominent trend lines from being anchored on statistically fragile estimates.

**Effect:** Balochistan's line is truncated after grade 7 (grades 8–10 have n_private = 44, 27, 22 respectively). National, Punjab, Sindh, and KPK are unaffected.

In [ ]:
CHART_PROVINCES = ['National', 'Punjab', 'Sindh', 'KPK', 'Balochistan']
PALETTE = {
    'National':    '#2c2c2c',
    'Punjab':      '#1f77b4',
    'Sindh':       '#d62728',
    'KPK':         '#2ca02c',
    'Balochistan': '#ff7f0e',
}
STYLES = {
    'National':    (None, 2.5),
    'Punjab':      ((6, 2), 1.8),
    'Sindh':       ((2, 2), 1.8),
    'KPK':         ((6, 2, 2, 2), 1.8),
    'Balochistan': ((1, 1), 1.8),
}

# Null out gap values where n_private is below the threshold
filtered = prov_df[prov_df['province'].isin(CHART_PROVINCES)].copy()
n_dropped = (filtered['n_private'] < THRESHOLD).sum()
filtered.loc[filtered['n_private'] < THRESHOLD, 'arithmetic_gap'] = np.nan
print(f'Data points suppressed (n_private < {THRESHOLD}): {n_dropped}')
print(filtered[filtered['arithmetic_gap'].isna()][['province', 'grade', 'n_private']])

fig, ax = plt.subplots(figsize=(9, 5))

for prov in CHART_PROVINCES:
    sub = filtered[filtered['province'] == prov].sort_values('grade')
    dash, lw = STYLES[prov]
    ax.plot(
        sub['grade'], sub['arithmetic_gap'],
        color=PALETTE[prov], linewidth=lw,
        dashes=dash if dash else (None, None),
        marker='o', markersize=4.5, markerfacecolor='white',
        markeredgewidth=1.4, label=prov,
    )

ax.axhline(0, color='#999999', linewidth=0.8, linestyle='--', zorder=0)

# Annotate the cutoff point for Balochistan
bal_last = filtered[(filtered['province'] == 'Balochistan') &
                     filtered['arithmetic_gap'].notna()].sort_values('grade').iloc[-1]
ax.annotate(
    f"  n < {THRESHOLD}\n  (grades 8–10)",
    xy=(bal_last['grade'], bal_last['arithmetic_gap']),
    xytext=(bal_last['grade'] + 0.3, bal_last['arithmetic_gap'] + 0.04),
    fontsize=8.5, color=PALETTE['Balochistan'],
    arrowprops=dict(arrowstyle='-', color=PALETTE['Balochistan'], lw=0.8),
)

ax.set_xlabel('Grade', labelpad=8)
ax.set_ylabel('Arithmetic gap\n(private − government)', labelpad=8)
ax.set_title('Govt–Private Arithmetic Gap by Grade  (n_private ≥ 50)',
             fontsize=13, fontweight='bold', pad=12)
ax.set_xticks(range(1, 11))
ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.2f'))
ax.legend(frameon=False, loc='upper right', fontsize=10)
sns.despine(ax=ax)
plt.tight_layout()

fig.savefig(FIGURES / '01b_gap_by_grade_province_filtered.png', bbox_inches='tight')
fig.savefig(FIGURES / '01b_gap_by_grade_province_filtered.pdf', bbox_inches='tight')
plt.show()